# 10 · Pydantic in Practice: A REST API with FastAPI + SQLAlchemy

**Module 15 – Pydantic: Data Validation & Settings** · Estimated time: 75 min

> Everything comes together: Pydantic models for input/output, SQLAlchemy for storage (Module 14 · 06), dependency injection, settings (lesson 09), error handling and tests (Module 07 · 04). We run the API **inside this notebook** with FastAPI's `TestClient` (no server needed).

`pip install "fastapi[standard]" sqlalchemy` (the `standard` extra includes uvicorn & the `fastapi` CLI).

### 🎯 What you will learn
- FastAPI basics: routes, path/query/body parameters validated by Pydantic
- The **schema pattern**: `Create` / `Update` / `Public` models
- `response_model` to filter output (never leak passwords!)
- SQLAlchemy sessions as **dependencies** (`Depends`) and `from_attributes`
- Status codes, `HTTPException`, automatic 422 validation errors
- `Annotated` parameter metadata: `Query`, `Path`, `Body`, `Field`
- Automatic **OpenAPI** docs from your models
- Testing with `TestClient` and dependency overrides
- Running for real with `fastapi dev` / uvicorn

In [1]:
import fastapi, pydantic, sqlalchemy
print("FastAPI", fastapi.__version__, "| Pydantic", pydantic.VERSION, "| SQLAlchemy", sqlalchemy.__version__)

FastAPI 0.151.0 | Pydantic 2.14.1 | SQLAlchemy 2.1.5


## 1. Database layer (SQLAlchemy 2.x)

In [2]:
from datetime import datetime
from sqlalchemy import create_engine, String, ForeignKey, func, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, sessionmaker, Session
from sqlalchemy.pool import StaticPool

engine = create_engine("sqlite://", connect_args={"check_same_thread": False}, poolclass=StaticPool)  # in-memory DB shared across threads
SessionLocal = sessionmaker(bind=engine, expire_on_commit=False)

class Base(DeclarativeBase):
    pass

class UserRow(Base):
    __tablename__ = "users"
    id: Mapped[int] = mapped_column(primary_key=True)
    email: Mapped[str] = mapped_column(String(200), unique=True, index=True)
    full_name: Mapped[str] = mapped_column(String(100))
    hashed_password: Mapped[str]
    is_active: Mapped[bool] = mapped_column(default=True)
    created_at: Mapped[datetime] = mapped_column(server_default=func.now())
    tasks: Mapped[list["TaskRow"]] = relationship(back_populates="owner", cascade="all, delete-orphan")

class TaskRow(Base):
    __tablename__ = "tasks"
    id: Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(String(200))
    done: Mapped[bool] = mapped_column(default=False)
    priority: Mapped[int] = mapped_column(default=3)
    owner_id: Mapped[int] = mapped_column(ForeignKey("users.id"))
    owner: Mapped[UserRow] = relationship(back_populates="tasks")

Base.metadata.create_all(engine)
print("tables:", list(Base.metadata.tables))

tables: ['users', 'tasks']


## 2. Pydantic schemas: Create / Update / Public

In [3]:
import hashlib
from typing import Annotated
from pydantic import BaseModel, ConfigDict, EmailStr, Field

class APIModel(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)

class UserCreate(APIModel):                    # what clients SEND
    email: EmailStr
    full_name: str = Field(min_length=2, max_length=100)
    password: str = Field(min_length=8)

class UserPublic(BaseModel):                   # what clients RECEIVE (no password!)
    model_config = ConfigDict(from_attributes=True)       # read from ORM objects
    id: int
    email: EmailStr
    full_name: str
    is_active: bool
    created_at: datetime

Priority = Annotated[int, Field(ge=1, le=5, description="1 = highest")]

class TaskCreate(APIModel):
    title: str = Field(min_length=1, max_length=200)
    priority: Priority = 3

class TaskUpdate(APIModel):                    # PATCH: everything optional
    title: str | None = Field(default=None, min_length=1, max_length=200)
    done: bool | None = None
    priority: Priority | None = None

class TaskPublic(BaseModel):
    model_config = ConfigDict(from_attributes=True)
    id: int
    title: str
    done: bool
    priority: int
    owner_id: int

class Page[T](BaseModel):
    items: list[T]
    total: int
    limit: int
    offset: int

def hash_password(pw: str) -> str:              # demo only: use argon2/bcrypt in real apps (Module 12 · 06)
    return hashlib.sha256(pw.encode()).hexdigest()

## 3. The FastAPI application

In [4]:
from collections.abc import Iterator
from fastapi import FastAPI, Depends, HTTPException, Query, Path, status
from sqlalchemy.exc import IntegrityError

app = FastAPI(title="Task API", version="1.0.0", description="A demo API built in the Python Mastery Course")

def get_db() -> Iterator[Session]:               # one session per request, always closed
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

DB = Annotated[Session, Depends(get_db)]

@app.get("/health")
def health() -> dict[str, str]:
    return {"status": "ok"}

@app.post("/users", response_model=UserPublic, status_code=status.HTTP_201_CREATED)
def create_user(user: UserCreate, db: DB):
    row = UserRow(email=user.email, full_name=user.full_name, hashed_password=hash_password(user.password))
    db.add(row)
    try:
        db.commit()
    except IntegrityError:
        db.rollback()
        raise HTTPException(status.HTTP_409_CONFLICT, detail="email already registered")
    db.refresh(row)
    return row                                    # an ORM object → converted by response_model

def get_user_or_404(db: Session, user_id: int) -> UserRow:
    row = db.get(UserRow, user_id)
    if row is None:
        raise HTTPException(status.HTTP_404_NOT_FOUND, detail=f"user {user_id} not found")
    return row

@app.get("/users/{user_id}", response_model=UserPublic)
def read_user(user_id: Annotated[int, Path(gt=0)], db: DB):
    return get_user_or_404(db, user_id)

@app.post("/users/{user_id}/tasks", response_model=TaskPublic, status_code=201)
def create_task(user_id: Annotated[int, Path(gt=0)], task: TaskCreate, db: DB):
    get_user_or_404(db, user_id)
    row = TaskRow(**task.model_dump(), owner_id=user_id)
    db.add(row); db.commit(); db.refresh(row)
    return row

@app.get("/users/{user_id}/tasks", response_model=Page[TaskPublic])
def list_tasks(
    user_id: Annotated[int, Path(gt=0)],
    db: DB,
    done: bool | None = None,
    max_priority: Annotated[int, Query(ge=1, le=5)] = 5,
    limit: Annotated[int, Query(ge=1, le=100)] = 20,
    offset: Annotated[int, Query(ge=0)] = 0,
):
    get_user_or_404(db, user_id)
    stmt = select(TaskRow).where(TaskRow.owner_id == user_id, TaskRow.priority <= max_priority)
    if done is not None:
        stmt = stmt.where(TaskRow.done == done)
    total = db.scalar(select(func.count()).select_from(stmt.subquery()))
    rows = db.scalars(stmt.order_by(TaskRow.priority, TaskRow.id).limit(limit).offset(offset)).all()
    return Page[TaskPublic](items=[TaskPublic.model_validate(r) for r in rows], total=total, limit=limit, offset=offset)

@app.patch("/tasks/{task_id}", response_model=TaskPublic)
def update_task(task_id: int, changes: TaskUpdate, db: DB):
    row = db.get(TaskRow, task_id)
    if row is None:
        raise HTTPException(404, detail="task not found")
    for field, value in changes.model_dump(exclude_unset=True).items():     # only fields the client sent
        setattr(row, field, value)
    db.commit(); db.refresh(row)
    return row

@app.delete("/tasks/{task_id}", status_code=204)
def delete_task(task_id: int, db: DB) -> None:
    row = db.get(TaskRow, task_id)
    if row is None:
        raise HTTPException(404, detail="task not found")
    db.delete(row); db.commit()

print([f"{sorted(r.methods)[0]:<6} {r.path}" for r in app.routes if hasattr(r, "methods") and r.path.startswith(("/users", "/tasks", "/health"))])

['GET    /health', 'POST   /users', 'GET    /users/{user_id}', 'POST   /users/{user_id}/tasks', 'GET    /users/{user_id}/tasks', 'PATCH  /tasks/{task_id}', 'DELETE /tasks/{task_id}']


Notice how little validation code there is: **types and Pydantic models are the validation**.

## 4. Calling the API with `TestClient`

In [5]:
from fastapi.testclient import TestClient
client = TestClient(app)

r = client.post("/users", json={"email": "asha@example.com", "full_name": "  Asha Rao ", "password": "s3cret-pass"})
print(r.status_code, r.json())                    # password is NOT in the response
user_id = r.json()["id"]

for title, prio in [("Write report", 2), ("Fix bug #42", 1), ("Lunch", 5)]:
    client.post(f"/users/{user_id}/tasks", json={"title": title, "priority": prio})

print(client.get(f"/users/{user_id}/tasks", params={"max_priority": 3}).json())

201 {'id': 1, 'email': 'asha@example.com', 'full_name': 'Asha Rao', 'is_active': True, 'created_at': '2026-10-20T14:16:09'}
{'items': [{'id': 2, 'title': 'Fix bug #42', 'done': False, 'priority': 1, 'owner_id': 1}, {'id': 1, 'title': 'Write report', 'done': False, 'priority': 2, 'owner_id': 1}], 'total': 2, 'limit': 20, 'offset': 0}


/private/tmp/claude-501/-Users-gaurvendrasingh-Desktop-python/f4ffaf20-c7ab-4c1c-ae64-d5055060b56d/scratchpad/venv315/lib/python3.15/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


In [6]:
task_id = client.get(f"/users/{user_id}/tasks").json()["items"][0]["id"]
print(client.patch(f"/tasks/{task_id}", json={"done": True}).json())             # partial update
print(client.get(f"/users/{user_id}/tasks", params={"done": "true"}).json()["total"])
print(client.delete(f"/tasks/{task_id}").status_code)

{'id': 2, 'title': 'Fix bug #42', 'done': True, 'priority': 1, 'owner_id': 1}
1
204


> ⚠️ The `StarletteDeprecationWarning` above comes from the library itself: recent Starlette versions want the **`httpx2`** package for `TestClient` (plain `httpx` still works for now). Install what the warning asks for in new projects.

## 5. Errors: validation (422), conflicts, not found

In [7]:
bad = client.post("/users", json={"email": "not-an-email", "full_name": "A", "password": "short", "is_admin": True})
print(bad.status_code)
for err in bad.json()["detail"]:
    print(" ", err["loc"], "→", err["msg"])

print(client.post("/users", json={"email": "asha@example.com", "full_name": "Dup", "password": "12345678"}).json())
print(client.get("/users/999").status_code, client.get("/users/999").json())
print(client.get("/users/0").status_code)                                         # Path(gt=0)
print(client.get(f"/users/{user_id}/tasks", params={"limit": 1000}).json()["detail"][0]["msg"])

422
  ['body', 'email'] → value is not a valid email address: An email address must have an @-sign.
  ['body', 'full_name'] → String should have at least 2 characters
  ['body', 'password'] → String should have at least 8 characters
  ['body', 'is_admin'] → Extra inputs are not permitted
{'detail': 'email already registered'}
404 {'detail': 'user 999 not found'}
422
Input should be less than or equal to 100


FastAPI turns every Pydantic `ValidationError` on the request into a **422 Unprocessable Content** response listing each problem with its location (`body`, `query`, `path`).

## 6. Automatic OpenAPI documentation

In [8]:
spec = client.get("/openapi.json").json()
print(spec["info"])
print(list(spec["paths"]))
print(list(spec["components"]["schemas"])[:8])
print(spec["components"]["schemas"]["UserCreate"]["properties"]["password"])

{'title': 'Task API', 'description': 'A demo API built in the Python Mastery Course', 'version': '1.0.0'}
['/health', '/users', '/users/{user_id}', '/users/{user_id}/tasks', '/tasks/{task_id}']
['HTTPValidationError', 'Page_TaskPublic_', 'TaskCreate', 'TaskPublic', 'TaskUpdate', 'UserCreate', 'UserPublic', 'ValidationError']
{'type': 'string', 'minLength': 8, 'title': 'Password'}


Running the app for real (`fastapi dev main.py`) gives interactive docs at **`/docs`** (Swagger UI) and **`/redoc`**, generated from these same Pydantic models.

## 7. Testing with dependency overrides

Swap the real DB for a fresh one in tests, without touching application code:

In [9]:
test_engine = create_engine("sqlite://", connect_args={"check_same_thread": False}, poolclass=StaticPool)
Base.metadata.create_all(test_engine)
TestSession = sessionmaker(bind=test_engine)

def get_test_db():
    db = TestSession()
    try:
        yield db
    finally:
        db.close()

app.dependency_overrides[get_db] = get_test_db
r = client.get(f"/users/{user_id}")
print("user from the real DB visible in the test DB?", r.status_code)        # 404: isolated
app.dependency_overrides.clear()

user from the real DB visible in the test DB? 404


In a pytest suite this goes into a `conftest.py` fixture (Module 07 · 04).

## 8. Settings via dependency (lesson 09)

```python
from functools import lru_cache
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    database_url: str = "sqlite:///app.db"
    page_size_max: int = 100

@lru_cache
def get_settings() -> Settings:
    return Settings()

@app.get("/config")
def config(settings: Annotated[Settings, Depends(get_settings)]):
    return {"page_size_max": settings.page_size_max}
```

## 9. Running it for real

Project layout (Module 07 · 05, Module 10 · 08):
```
task_api/
├── pyproject.toml          # uv add "fastapi[standard]" sqlalchemy pydantic-settings
└── src/task_api/
    ├── main.py             # app = FastAPI(); include_router(...)
    ├── db.py               # engine, SessionLocal, get_db
    ├── models.py           # SQLAlchemy ORM
    ├── schemas.py          # Pydantic models
    ├── routers/users.py    # APIRouter for /users
    └── settings.py
```
```bash
uv run fastapi dev src/task_api/main.py                   # auto-reload, http://127.0.0.1:8000/docs
uv run uvicorn task_api.main:app --host 0.0.0.0 --port 8000 --workers 4   # production
```
Async version: `async def` endpoints with `sqlalchemy.ext.asyncio` (Module 14 · 06) and `httpx.AsyncClient` for outgoing calls.

## 📝 Summary
- FastAPI validates path/query/body parameters with Pydantic from type hints (`Annotated[int, Query(ge=1)]`).
- Use separate **Create / Update / Public** schemas; `response_model` filters output; `from_attributes` reads ORM rows.
- Inject sessions/settings with `Depends`; override dependencies in tests.
- Validation failures become 422 responses automatically; OpenAPI docs come for free.

## 🏋️ Practice
1. Add `GET /users` with pagination and an email search filter (`?q=asha`).
2. Add a `tags: list[str]` field to tasks (max 5, each 1–20 chars), stored as a comma-separated string or a separate table.
3. Write pytest tests for create/update/delete using a `client` fixture with an overridden database.
4. Convert the endpoints to `async def` with an async SQLAlchemy engine (`sqlite+aiosqlite`).